                # BootstrapFewShot

                Run a teacher over training examples, keep successful traces under the metric, and use them as demonstrations.

                **Use it when:** Labels exist but worked reasoning traces may teach the student more than labels alone.

                **What compilation changes:** Adds up to two bootstrapped and two labeled demonstrations; it does not search instruction text.

                Important configuration in this benchmark:

                - `max_bootstrapped_demos=2`
- `max_labeled_demos=2`
- one bootstrap round and one tolerated error

                The 74-row dataset and pair-grouped train/validation/test membership are frozen.
                The test partition is deliberately baseline-adversarial, so these scores teach
                optimizer tradeoffs; they are not a general-purpose AI-detector leaderboard.

In [1]:
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
REPO_ROOT = cwd if (cwd / "chapter06").is_dir() else cwd.parent
if not (REPO_ROOT / "chapter06" / "results" / "benchmark_summary.json").exists():
    raise RuntimeError("Run this notebook from the repository or chapter06 directory.")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from chapter06.notebook_support import (
    artifact_paths,
    benchmark_snapshot,
    learned_program_preview,
    verify_prompt_artifact,
)

OPTIMIZER = 'bootstrap-few-shot'
print(f"optimizer={OPTIMIZER!r}")
print("reading the checked-in chapter result; no API calls")

optimizer='bootstrap-few-shot'
reading the checked-in chapter result; no API calls


                ## Compile shape

                This is the essential DSPy call used by the shared runner (setup variables omitted):

                ```python
                optimizer = dspy.BootstrapFewShot(
    metric=exact_match, max_bootstrapped_demos=2,
    max_labeled_demos=2, max_rounds=1, max_errors=1,
)
optimized_detector = optimizer.compile(detector, trainset=trainset)
                ```

                `compile` returns a program. Calling that program on the untouched test examples is
                a separate phase; the notebook reports optimization cost/time separately from inference latency.

In [2]:
print(benchmark_snapshot(OPTIMIZER))
print()
print(artifact_paths(OPTIMIZER))

BootstrapFewShot — frozen full-profile rerun
status: completed
task model: openai/gpt-5.6-luna
test accuracy: 75.0%
uplift: +25.0 points vs Luna reference
optimization: $0.0029 and 4.3s
inference latency: mean 1.94s; p95 3.14s
reload checks: prompt=True; model=None; predictions=3/3 labels

Published artifacts:
- canonical program snapshot: chapter06/optimized_programs/final/bootstrap-few-shot.json
- canonical prompt: chapter06/results/final_prompts/bootstrap-few-shot.json
- chapter comparison: chapter06/CHAPTER_RESULTS.md


## Read the result

Compare its demos with LabeledFewShot: bootstrapped examples include model-produced reasoning that passed the exact-match metric.

The next cell shows a bounded readable preview. The complete, lossless prompt and
saved program snapshot remain at the paths printed above.

In [3]:
print(learned_program_preview(OPTIMIZER))
print()
print("Frozen program/prompt consistency:", verify_prompt_artifact(OPTIMIZER))

Predictor: detect.predict
Learned instruction (59 characters):
Decide whether the supplied passage was generated by an AI.

Demonstrations: 2
1. is_ai=False: We've covered the "CR" of CRUD. Now let's move on to the "U" (Update). Updating a resource is very similar to creating a resource. They are both multi-step processes. First, the…
2. is_ai=True: Having covered the “CR” in CRUD, we now turn to “U” for Update. Updating a resource closely resembles creating one, for both are multi-step affairs. First, the user requests a f…

Frozen program/prompt consistency: {'checked': True, 'predictors': 1, 'prompt_state_equal': True, 'mismatches': []}


## Apply the pattern

Adapt the compile shape above to your own DSPy program, metric, and frozen
train/validation split. Evaluate the returned program on a test set that was not
used during compilation, and compare accuracy, compile cost, and inference latency
rather than treating a single score as the whole result.

The complete Chapter 6 rerun is summarized in `CHAPTER_RESULTS.md`. Raw provider
transcripts and temporary training outputs are intentionally excluded from the
student download.